# LeetCode 3226 — Number of Bit Changes to Make Two Integers Equal bằng Python

Link đề: https://leetcode.com/problems/number-of-bit-changes-to-make-two-integers-equal/ · Độ khó: **Easy** · Chủ đề: Bit Manipulation

Skeleton LeetCode:

```python
class Solution:
    def minChanges(self, n: int, k: int) -> int:
        ...
```

Ghi chú: mọi đáp án trong notebook đã được kiểm chứng bằng code (kể cả vét cạn bằng BFS cho mọi `n, k` nhỏ). Ba ví dụ chính lấy từ đề; những ví dụ tự dựng thêm có ghi rõ "tự dựng" — nếu muốn khớp từng chữ thì nên đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: được phép chọn **một bit bằng `1`** trong dạng nhị phân của `n` và **đổi nó thành `0`**; đếm số lần đổi cần để `n` thành đúng `k`, hoặc `-1` nếu không thể.
2. Nắm các công cụ Python về bit: `bin()`, `&`, `^`, `>>`, kiểm tra bit thứ `i` bằng `(n >> i) & 1`, đếm bit `1` bằng `bin(x).count("1")` và `int.bit_count()`, mẹo `x & (x - 1)` xóa bit `1` thấp nhất.
3. Đi từ cách **duyệt từng bit** → **kiểm tra `n & k == k` rồi đếm bit của `n ^ k`** → **`bit_count`** → **vòng lặp Brian Kernighan** → **BFS mô phỏng** (chỉ để kiểm chứng).
4. Hiểu mấu chốt: `k` đạt được từ `n` **khi và chỉ khi** các bit `1` của `k` là **tập con** các bit `1` của `n` (`n & k == k`); khi đó các bit cần đổi chính là các bit `1` của `n ^ k`, nên đáp án là số bit `1` của `n ^ k`.
5. Tránh các lỗi kinh điển: chỉ đếm bit `1` của `n ^ k` mà quên kiểm tra khả thi, kiểm tra ngược chiều (`n & k == n`), so sánh `popcount(n) − popcount(k)`, và quên rằng không được đổi `0` thành `1`.

## 2. Đề bài & ví dụ

Cho hai số nguyên dương `n` và `k`.

Bạn có thể chọn **bất kỳ** bit nào trong **biểu diễn nhị phân** của `n` mà bằng `1` rồi đổi nó thành `0`. Làm lặp lại tùy ý.

Trả về **số lần đổi** cần thiết để `n` bằng `k`. Nếu không thể, trả về `-1`.

**Ví dụ 1**

- Input: `n = 13, k = 4`
- Output: `2`
- Giải thích: `n = (1101)₂`, `k = (0100)₂`. Đổi bit thứ nhất (cao nhất) và bit thứ tư (thấp nhất) của `n` từ `1` sang `0` được `(0100)₂ = 4 = k`.

**Ví dụ 2**

- Input: `n = 21, k = 21`
- Output: `0`
- Giải thích: `n` và `k` đã bằng nhau, không cần đổi.

**Ví dụ 3**

- Input: `n = 14, k = 13`
- Output: `-1`
- Giải thích: `14 = (1110)₂`, `13 = (1101)₂`. Bit thấp nhất của `k` là `1` nhưng của `n` là `0`, mà ta không thể biến `0` thành `1` ⇒ không thể.

**Ràng buộc:** `1 <= n, k <= 10^6`.

In [1]:
examples = [
    (13, 4, 2),
    (21, 21, 0),
    (14, 13, -1),
]
for n, k, expected in examples:
    print(f"n={n}, k={k} -> kỳ vọng {expected}")

n=13, k=4 -> kỳ vọng 2
n=21, k=21 -> kỳ vọng 0
n=14, k=13 -> kỳ vọng -1


## 3. Kiến thức nền

### 3.1. Dạng nhị phân: `bin()` và `format`

`bin(13)` cho chuỗi `'0b1101'` (tiền tố `0b`). Muốn chuỗi nhị phân căn đều độ rộng để so sánh từng cột, dùng `format(13, "08b")` hoặc f-string `f"{13:08b}"`: `08` nghĩa là rộng `8` ký tự, thêm số `0` ở đầu.

In [2]:
print(bin(13), bin(4))                       # 0b1101 0b100
print(format(13, "08b"), format(4, "08b"))   # 00001101 00000100
print(f"{21:b}", int("10101", 2))            # 10101 21

0b1101 0b100
00001101 00000100
10101 21


### 3.2. Các phép bit: `&`, `|`, `^`

Áp dụng **từng cột** của hai số nhị phân:

- `a & b` (AND): bit kết quả `1` khi **cả hai** bit là `1`.
- `a | b` (OR): `1` khi **ít nhất một** bit là `1`.
- `a ^ b` (XOR): `1` khi hai bit **khác nhau**.

Trong Python, `&`, `^`, `|` có độ ưu tiên **cao hơn** phép so sánh (`==`, `!=`), nhưng người mới nên **luôn thêm ngoặc** cho dễ đọc.

In [3]:
n, k = 13, 4
print(f"n     = {n:04b}")
print(f"k     = {k:04b}")
print(f"n & k = {n & k:04b} ({n & k})")
print(f"n | k = {n | k:04b} ({n | k})")
print(f"n ^ k = {n ^ k:04b} ({n ^ k})")

n     = 1101
k     = 0100
n & k = 0100 (4)
n | k = 1101 (13)
n ^ k = 1001 (9)


### 3.3. "Tập con" của các bit: `n & k == k`

Xem mỗi số như một **tập các vị trí bit `1`**. `k` là **tập con** của `n` khi mọi bit `1` của `k` cũng là bit `1` của `n`. Khi đó `n & k` giữ đúng các bit chung — chính là `k`. Ngược lại, nếu `n & k != k` thì có bit `1` của `k` mà `n` không có.

In [4]:
for n, k in [(13, 4), (21, 21), (14, 13)]:
    print(f"n={n:05b}, k={k:05b}, n&k={n & k:05b} ->", "k là tập con của n" if (n & k) == k else "KHÔNG là tập con")

n=01101, k=00100, n&k=00100 -> k là tập con của n
n=10101, k=10101, n&k=10101 -> k là tập con của n
n=01110, k=01101, n&k=01100 -> KHÔNG là tập con


### 3.4. Đếm số bit `1` (popcount)

Hai cách quen thuộc: `bin(x).count("1")` (đếm ký tự `'1'` trong chuỗi nhị phân) và `x.bit_count()` (Python `>= 3.10`, nhanh hơn, không cần tạo chuỗi). `0` có `0` bit `1`.

In [5]:
for x in [0, 1, 5, 13, 255, 10**6]:
    print(x, bin(x), bin(x).count("1"), x.bit_count())
    assert bin(x).count("1") == x.bit_count()

0 0b0 0 0
1 0b1 1 1
5 0b101 2 2
13 0b1101 3 3
255 0b11111111 8 8
1000000 0b11110100001001000000 7 7


### 3.5. Lấy bit thứ `i`: dịch bit `>>` và `& 1`

`n >> i` dịch phải `i` vị trí (bỏ `i` bit thấp nhất); `(n >> i) & 1` cho **bit thứ `i`** (đếm từ `0` ở bên phải). Ví dụ `13 = 1101`: bit `0` là `1`, bit `1` là `0`, bit `2` là `1`, bit `3` là `1`.

In [6]:
n = 13
print([(n >> i) & 1 for i in range(4)])        # [1, 0, 1, 1]  (bit 0 ở đầu)
print([(n >> i) & 1 for i in reversed(range(4))])   # [1, 1, 0, 1]  đọc theo thứ tự viết nhị phân

[1, 0, 1, 1]
[1, 1, 0, 1]


### 3.6. Mẹo Brian Kernighan: `x & (x - 1)`

Trừ `1` làm bit `1` thấp nhất đổi thành `0` và các bit thấp hơn đổi thành `1`; khi AND với `x` ban đầu, **bit `1` thấp nhất bị xóa**, phần còn lại giữ nguyên. Lặp cho tới khi `x = 0`: số vòng lặp = số bit `1`.

In [7]:
x = 0b101100
steps = 0
while x:
    print(f"{x:06b} -> {x & (x - 1):06b}")
    x &= x - 1
    steps += 1
print("số bit 1:", steps)       # 3

101100 -> 101000
101000 -> 100000
100000 -> 000000
số bit 1: 3


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Ví dụ 1: `n = 13 = 1101`, `k = 4 = 0100`. Đặt hai số thẳng cột rồi xét **từng bit**:

| bit | `n` | `k` | nhận xét |
|---|---|---|---|
| 3 | 1 | 0 | phải đổi `1 → 0` (cần 1 lần) |
| 2 | 1 | 1 | giữ nguyên |
| 1 | 0 | 0 | giữ nguyên |
| 0 | 1 | 0 | phải đổi `1 → 0` (cần 1 lần) |

Tổng `2` lần. Ví dụ 3: `n = 14 = 1110`, `k = 13 = 1101`:

| bit | `n` | `k` | nhận xét |
|---|---|---|---|
| 3 | 1 | 1 | giữ |
| 2 | 1 | 1 | giữ |
| 1 | 1 | 0 | đổi `1 → 0` |
| 0 | 0 | 1 | **cần `0 → 1`: không được phép** ⇒ `-1` |

Quy tắc theo từng cột: `(n, k) = (1, 0)` ⇒ cần 1 lần đổi; `(0, 1)` ⇒ **bất khả thi**; `(0, 0)` hoặc `(1, 1)` ⇒ không làm gì. Ta chỉ cần quét mọi cột.

In [8]:
n, k = 13, 4
changes = 0
for i in reversed(range(4)):
    a, b = (n >> i) & 1, (k >> i) & 1
    note = "đổi 1->0" if (a, b) == (1, 0) else ("BẤT KHẢ THI" if (a, b) == (0, 1) else "giữ nguyên")
    print(f"bit {i}: n={a}, k={b} -> {note}")
    changes += (a, b) == (1, 0)
print("tổng số lần đổi:", changes)

bit 3: n=1, k=0 -> đổi 1->0
bit 2: n=1, k=1 -> giữ nguyên
bit 1: n=0, k=0 -> giữ nguyên
bit 0: n=1, k=0 -> đổi 1->0
tổng số lần đổi: 2


## 5. Cách 1 — Brute Force / baseline: duyệt từng bit

**Ý tưởng.** `n, k <= 10^6 < 2^20`, nên chỉ cần xét `20` bit. Với mỗi vị trí `i`:

- Lấy `a = (n >> i) & 1`, `b = (k >> i) & 1`.
- `a = 0, b = 1` ⇒ trả `-1` ngay.
- `a = 1, b = 0` ⇒ tăng đếm.

Kết quả là số lần đổi. Đây đúng là cách làm tay ở mục 4 viết thành vòng lặp.

In [9]:
def min_changes_bitwise(n: int, k: int) -> int:
    changes = 0
    for i in range(20):                       # 10^6 < 2^20 nên đủ 20 bit
        a = (n >> i) & 1                      # bit thứ i của n
        b = (k >> i) & 1                      # bit thứ i của k
        if a == 0 and b == 1:                 # cần biến 0 thành 1: không được
            return -1
        if a == 1 and b == 0:                 # phải biến 1 thành 0
            changes += 1
    return changes


for n, k, expected in examples:
    print(min_changes_bitwise(n, k), "(kỳ vọng", expected, ")")

2 (kỳ vọng 2 )
0 (kỳ vọng 0 )
-1 (kỳ vọng -1 )


**Độ phức tạp (Cách 1).** Số bit `B = 20` (cỡ `log n`) ⇒ **Time `O(log n)`**, **Space `O(1)`**.

**Nút thắt:** vòng lặp từng bit đúng nhưng dài. Hai việc "có bất khả thi không" và "đếm số bit cần đổi" đều có thể gom thành **phép toán bit trên cả số** (không cần tách từng bit).

## 6. Ý tưởng thuật toán / bước đột phá

Theo quy tắc từng cột ở mục 4:

- **Bất khả thi** ⇔ tồn tại cột `(n, k) = (0, 1)` ⇔ `k` có bit `1` mà `n` không có ⇔ **`n & k != k`**.
- Nếu khả thi, mọi cột còn lại là `(0,0)`, `(1,1)` hoặc `(1,0)`. Cột cần đổi là `(1,0)`, tức hai bit **khác nhau** ⇒ nằm trong **`n ^ k`** (cột `(1,1)`, `(0,0)` cho `0`; `(0,1)` đã bị loại). Số cột cần đổi chính là **số bit `1` của `n ^ k`**.

Vậy: `-1 if n & k != k else popcount(n ^ k)`.

In [10]:
for n, k, expected in examples:
    ok = (n & k) == k
    x = n ^ k
    print(f"n={n:05b} k={k:05b}  n&k==k? {ok}  n^k={x:05b}  popcount={x.bit_count()}  -> ", x.bit_count() if ok else -1)

n=01101 k=00100  n&k==k? True  n^k=01001  popcount=2  ->  2
n=10101 k=10101  n&k==k? True  n^k=00000  popcount=0  ->  0
n=01110 k=01101  n&k==k? False  n^k=00011  popcount=2  ->  -1


## 7. Các cách tiếp theo

### Cách 2 — `n & k` kiểm tra, rồi đếm `'1'` trong `bin(n ^ k)`

**Giải thích từng đoạn.**

1. `if (n & k) != k: return -1` — khử trường hợp bất khả thi.
2. `n ^ k` — chỉ giữ các cột khác nhau (lúc này chỉ còn cột `(1,0)`).
3. `bin(...).count("1")` — đếm số ký tự `'1'` = số bit `1`.

In [11]:
def min_changes_bin_count(n: int, k: int) -> int:
    if (n & k) != k:                            # k có bit 1 mà n không có
        return -1
    return bin(n ^ k).count("1")                # đếm bit 1 của n ^ k


for n, k, expected in examples:
    print(min_changes_bin_count(n, k), "(kỳ vọng", expected, ")")

2 (kỳ vọng 2 )
0 (kỳ vọng 0 )
-1 (kỳ vọng -1 )


In [12]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionBinCount:
    def minChanges(self, n: int, k: int) -> int:
        if (n & k) != k:
            return -1
        return bin(n ^ k).count("1")


print(SolutionBinCount().minChanges(13, 4), SolutionBinCount().minChanges(14, 13))      # 2 -1

2 -1


**Độ phức tạp Cách 2:** **Time `O(log n)`** (tạo chuỗi nhị phân độ dài `~log n`), **Space `O(log n)`** cho chuỗi (coi như `O(1)` vì `<= 21` ký tự).

### Cách 3 — `int.bit_count()` (cách nộp gọn nhất)

**Giải thích.** Từ Python `3.10`, `x.bit_count()` đếm bit `1` trực tiếp (cài đặt bằng lệnh phần cứng `popcount`), không cần tạo chuỗi. Toàn bộ lời giải thành **một dòng**.

In [13]:
def min_changes_bit_count(n: int, k: int) -> int:
    return -1 if (n & k) != k else (n ^ k).bit_count()


for n, k, expected in examples:
    print(min_changes_bit_count(n, k), "(kỳ vọng", expected, ")")

2 (kỳ vọng 2 )
0 (kỳ vọng 0 )
-1 (kỳ vọng -1 )


In [14]:
# Code nộp LeetCode cho Cách 3
class SolutionBitCount:
    def minChanges(self, n: int, k: int) -> int:
        return -1 if n & k != k else (n ^ k).bit_count()


print(SolutionBitCount().minChanges(21, 21), SolutionBitCount().minChanges(13, 4))      # 0 2

0 2


**Độ phức tạp Cách 3:** **Time `O(1)`** thực tế (một số `<= 20` bit nằm gọn trong một từ máy), về lý thuyết `O(log n)`; **Space `O(1)`**.

### Cách 4 — Vòng lặp Brian Kernighan trên `n ^ k`

**Giải thích.** Không dùng `bit_count`/`bin`: tự đếm bằng `x &= x - 1` (mỗi lần xóa đúng một bit `1` thấp nhất). Số vòng lặp bằng số bit `1`. Cách này cũng hay xuất hiện ở bài **Number of 1 Bits (191)**.

In [15]:
def min_changes_kernighan(n: int, k: int) -> int:
    if (n & k) != k:
        return -1
    x = n ^ k
    count = 0
    while x:
        x &= x - 1                  # xóa bit 1 thấp nhất
        count += 1
    return count


for n, k, expected in examples:
    print(min_changes_kernighan(n, k), "(kỳ vọng", expected, ")")

2 (kỳ vọng 2 )
0 (kỳ vọng 0 )
-1 (kỳ vọng -1 )


**Độ phức tạp Cách 4:** **Time `O(số bit 1 của n ^ k)` ≤ `O(log n)`**, **Space `O(1)`**.

### Cách 5 — BFS mô phỏng đúng luật (chỉ để kiểm chứng, `n` nhỏ)

**Giải thích.** Coi mỗi giá trị của `n` là một **nút**; một lần đổi là một **cạnh** sang giá trị mới (xóa một bit `1`). BFS từ `n` tìm số cạnh ít nhất tới `k`. Không có đường đi ⇒ `-1`. Cách này **không dựa vào bất kỳ nhận xét thông minh nào**, nên là "thước đo" tin cậy để kiểm tra công thức. Nhưng số nút có thể tới `2^(số bit 1)`, nên chỉ chạy được khi `n` nhỏ.

In [16]:
from collections import deque


def min_changes_bfs(n: int, k: int) -> int:
    dist = {n: 0}
    queue = deque([n])
    while queue:
        cur = queue.popleft()
        if cur == k:
            return dist[cur]
        for i in range(cur.bit_length()):
            if (cur >> i) & 1:                     # bit i đang là 1 -> được phép đổi thành 0
                nxt = cur & ~(1 << i)
                if nxt not in dist:
                    dist[nxt] = dist[cur] + 1
                    queue.append(nxt)
    return -1


for n, k, expected in examples:
    print(min_changes_bfs(n, k), "(kỳ vọng", expected, ")")

2 (kỳ vọng 2 )
0 (kỳ vọng 0 )
-1 (kỳ vọng -1 )


**Độ phức tạp Cách 5:** **Time `O(2^p · log n)`** với `p` là số bit `1` của `n` (tối đa `2^20` nút — quá chậm để nộp), **Space `O(2^p)`**. Vì vậy ta chỉ dùng cho `n` nhỏ trong phần test.

## 8. TRACE từng bước

Ta trace Cách 1 (từng bit) trên `n = 13, k = 4`, Cách 4 (Kernighan) trên `n = 29, k = 9` (tự dựng), và một ca bất khả thi `n = 14, k = 13` để thấy nhánh `-1`.

In [17]:
def trace_bitwise(n: int, k: int) -> int:
    print(f"n = {n} = {n:06b}, k = {k} = {k:06b}")
    changes = 0
    for i in reversed(range(6)):
        a, b = (n >> i) & 1, (k >> i) & 1
        if a == 0 and b == 1:
            print(f"  bit {i}: n={a}, k={b} -> bất khả thi, trả -1")
            return -1
        if a == 1 and b == 0:
            changes += 1
            print(f"  bit {i}: n={a}, k={b} -> đổi 1->0, đếm = {changes}")
        else:
            print(f"  bit {i}: n={a}, k={b} -> giữ nguyên")
    print("  kết quả:", changes)
    return changes


assert trace_bitwise(13, 4) == 2
assert trace_bitwise(14, 13) == -1

n = 13 = 001101, k = 4 = 000100
  bit 5: n=0, k=0 -> giữ nguyên
  bit 4: n=0, k=0 -> giữ nguyên
  bit 3: n=1, k=0 -> đổi 1->0, đếm = 1
  bit 2: n=1, k=1 -> giữ nguyên
  bit 1: n=0, k=0 -> giữ nguyên
  bit 0: n=1, k=0 -> đổi 1->0, đếm = 2
  kết quả: 2
n = 14 = 001110, k = 13 = 001101
  bit 5: n=0, k=0 -> giữ nguyên
  bit 4: n=0, k=0 -> giữ nguyên
  bit 3: n=1, k=1 -> giữ nguyên
  bit 2: n=1, k=1 -> giữ nguyên
  bit 1: n=1, k=0 -> đổi 1->0, đếm = 1
  bit 0: n=0, k=1 -> bất khả thi, trả -1


In [18]:
def trace_kernighan(n: int, k: int) -> int:
    print(f"n = {n:06b}, k = {k:06b}, n & k = {n & k:06b}")
    if (n & k) != k:
        print("  n & k != k -> -1")
        return -1
    x = n ^ k
    print(f"  x = n ^ k = {x:06b}")
    count = 0
    while x:
        nxt = x & (x - 1)
        count += 1
        print(f"  x = {x:06b}, x-1 = {x - 1:06b}, x & (x-1) = {nxt:06b}, đếm = {count}")
        x = nxt
    print("  x = 0 -> dừng, kết quả:", count)
    return count


assert trace_kernighan(29, 9) == 2
assert trace_kernighan(63, 0 + 1) == 5

n = 011101, k = 001001, n & k = 001001
  x = n ^ k = 010100
  x = 010100, x-1 = 010011, x & (x-1) = 010000, đếm = 1
  x = 010000, x-1 = 001111, x & (x-1) = 000000, đếm = 2
  x = 0 -> dừng, kết quả: 2
n = 111111, k = 000001, n & k = 000001
  x = n ^ k = 111110
  x = 111110, x-1 = 111101, x & (x-1) = 111100, đếm = 1
  x = 111100, x-1 = 111011, x & (x-1) = 111000, đếm = 2
  x = 111000, x-1 = 110111, x & (x-1) = 110000, đếm = 3
  x = 110000, x-1 = 101111, x & (x-1) = 100000, đếm = 4
  x = 100000, x-1 = 011111, x & (x-1) = 000000, đếm = 5
  x = 0 -> dừng, kết quả: 5


Bảng trace Cách 1 với `n = 13, k = 4`:

| bit `i` | `n` | `k` | hành động | đếm |
|---|---|---|---|---|
| 3 | 1 | 0 | đổi `1 → 0` | 1 |
| 2 | 1 | 1 | giữ | 1 |
| 1 | 0 | 0 | giữ | 1 |
| 0 | 1 | 0 | đổi `1 → 0` | 2 |

Kết quả `2`. Với `n = 29 = 11101`, `k = 9 = 01001`: `n & k = 01001 = k` (khả thi), `n ^ k = 10100` có `2` bit `1` ⇒ `2` (khớp trace Kernighan).

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (`bit_count`): ngắn, `O(1)` thực tế. Nếu môi trường chấm dùng Python cũ hơn `3.10`, thay bằng `bin(n ^ k).count("1")` (Cách 2).

In [19]:
class Solution:
    def minChanges(self, n: int, k: int) -> int:
        if (n & k) != k:                # k có bit 1 mà n không có: không thể (chỉ được đổi 1 -> 0)
            return -1
        return (n ^ k).bit_count()      # các bit khác nhau chính là các bit 1 cần đổi thành 0


sol = Solution()
for n, k, expected in examples:
    print(n, k, "->", sol.minChanges(n, k))

13 4 -> 2
21 21 -> 0
14 13 -> -1


## 10. Vì sao thuật toán đúng

- **Điều kiện cần.** Thao tác chỉ biến `1` thành `0`, nên bit `0` của `n` mãi là `0`. Nếu `k` có bit `1` tại vị trí mà `n` có `0` thì không bao giờ khớp ⇒ `-1`. Điều này chính là `n & k != k`.
- **Điều kiện đủ.** Nếu `n & k == k`, mỗi bit `1` của `k` đều có trong `n`. Các bit `1` thừa của `n` (không có trong `k`) ta xóa từng cái một ⇒ khớp `k`.
- **Tối thiểu.** Mỗi lần đổi chỉ sửa **một** bit, mà ta phải sửa **mỗi** vị trí khác nhau ít nhất một lần ⇒ cần ít nhất (số vị trí khác nhau) lần; ta đạt đúng con số đó (mỗi vị trí đúng một lần). Số vị trí khác nhau = `popcount(n ^ k)`.
- **Khi khả thi, `n ^ k` chỉ chứa cột `(1,0)`.** Vì cột `(0,1)` bị loại và cột bằng nhau cho `0` ⇒ mọi bit `1` của `n ^ k` đều là bit cần xóa.
- **Đối chiếu độc lập.** BFS (Cách 5) không dùng nhận xét nào và được so với công thức trên mọi cặp `1..255` ở mục 12.

## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Duyệt từng bit | `O(log n)` | `O(1)` |
| 2. `bin(...).count("1")` | `O(log n)` | `O(log n)` (chuỗi ngắn) |
| 3. `int.bit_count()` | `O(log n)` (thực tế hằng số) | `O(1)` |
| 4. Brian Kernighan | `O(p)`, `p` = số bit `1` của `n ^ k` | `O(1)` |
| 5. BFS mô phỏng | `O(2^q · log n)`, `q` = số bit `1` của `n` | `O(2^q)` |

Với `n, k <= 10^6` (tối đa `20` bit), các Cách 1–4 chạy tức thì.

## 12. Bộ test

Gồm: ví dụ đề, biên (`n = k`, `k = 1` với `n` chẵn, `k > n`, số lớn `10^6`), phản ví dụ cho công thức sai, đối chiếu BFS cho mọi `1 <= n, k <= 255`, và test ngẫu nhiên có seed trong ràng buộc.

In [20]:
import random

impls = {
    "bitwise": min_changes_bitwise,
    "bin_count": min_changes_bin_count,
    "bit_count": min_changes_bit_count,
    "kernighan": min_changes_kernighan,
    "submit": Solution().minChanges,
}

# 1) Ví dụ của đề
for n, k, expected in examples:
    for name, fn in impls.items():
        assert fn(n, k) == expected, (name, n, k)
    assert min_changes_bfs(n, k) == expected
print("PASS ví dụ đề")

# 2) Biên (tự dựng; đã đối chiếu bằng BFS hoặc suy luận tay)
edge = [
    (1, 1, 0),                     # nhỏ nhất, bằng nhau
    (1, 2, -1),                    # k có bit 1 mà n không có
    (2, 1, -1),                    # 10 -> 01 cần 0->1
    (3, 1, 1),                     # 11 -> 01
    (3, 2, 1),                     # 11 -> 10
    (7, 1, 2),                     # 111 -> 001
    (8, 7, -1),                    # 1000 -> 0111: k lớn hơn nhưng không là tập con
    (10**6, 10**6, 0),             # số lớn, bằng nhau
    (2**19 + 2**18, 2**18, 1),     # 786432 -> 262144
    (2**19, 1, -1),
]
for n, k, expected in edge:
    for name, fn in impls.items():
        assert fn(n, k) == expected, (name, n, k, expected)
print("PASS biên")

PASS ví dụ đề
PASS biên


In [21]:
# 3) Phản ví dụ cho các công thức "nghe có lý" nhưng sai
def sai_chi_xor(n, k):
    return (n ^ k).bit_count()                       # quên kiểm tra khả thi

def sai_hieu_popcount(n, k):
    return n.bit_count() - k.bit_count()             # chỉ so số lượng bit 1, không so vị trí

for n, k in [(14, 13), (8, 7), (13, 4)]:
    print((n, k), "đúng:", Solution().minChanges(n, k), "| chỉ xor:", sai_chi_xor(n, k), "| hiệu popcount:", sai_hieu_popcount(n, k))
assert sai_chi_xor(14, 13) != Solution().minChanges(14, 13)           # 2 thay vì -1
assert sai_hieu_popcount(14, 13) != Solution().minChanges(14, 13)     # 0 thay vì -1

(14, 13) đúng: -1 | chỉ xor: 2 | hiệu popcount: 0
(8, 7) đúng: -1 | chỉ xor: 4 | hiệu popcount: -2
(13, 4) đúng: 2 | chỉ xor: 2 | hiệu popcount: 2


In [22]:
# 4) Đối chiếu BFS (mô phỏng thật) với mọi n, k trong 1..255
total = 0
for n in range(1, 256):
    # BFS một lần từ n để lấy khoảng cách tới mọi k
    dist = {n: 0}
    queue = deque([n])
    while queue:
        cur = queue.popleft()
        for i in range(cur.bit_length()):
            if (cur >> i) & 1:
                nxt = cur & ~(1 << i)
                if nxt not in dist:
                    dist[nxt] = dist[cur] + 1
                    queue.append(nxt)
    for k in range(1, 256):
        expected = dist.get(k, -1)
        for name, fn in impls.items():
            assert fn(n, k) == expected, (name, n, k)
        total += 1
print("PASS đối chiếu BFS", total, "cặp (n, k)")

# 5) Ngẫu nhiên có seed, đúng ràng buộc 1 <= n, k <= 10^6
random.seed(3226)
for _ in range(5000):
    n = random.randint(1, 10**6)
    if random.random() < 0.5:
        k = n & random.randint(1, 10**6)          # k thường là tập con của n
        k = k or 1
    else:
        k = random.randint(1, 10**6)
    ref = min_changes_bitwise(n, k)
    for name, fn in impls.items():
        assert fn(n, k) == ref, (name, n, k)
print("PASS 5000 test ngẫu nhiên")

PASS đối chiếu BFS 65025 cặp (n, k)


PASS 5000 test ngẫu nhiên


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chỉ đếm bit của `n ^ k`, quên kiểm tra khả thi

In [23]:
def sai_quen_kiem_tra(n, k):
    return (n ^ k).bit_count()                  # SAI: không phát hiện trường hợp cần đổi 0 -> 1


print("sai:", sai_quen_kiem_tra(14, 13), "| đúng:", Solution().minChanges(14, 13))
print("sai:", sai_quen_kiem_tra(13, 4), "| đúng:", Solution().minChanges(13, 4))

sai: 2 | đúng: -1
sai: 2 | đúng: 2


**Vì sao sai:** `n ^ k` đếm **mọi** cột khác nhau, kể cả cột `(0, 1)` (cần đổi `0` thành `1`, không được phép). `14 ^ 13 = 0011` có `2` bit `1` nên cho `2`, trong khi đáp án là `-1`. **Cách sửa:** kiểm tra `n & k == k` trước.

### Lỗi 2: Kiểm tra ngược chiều (`n & k == n`)

In [24]:
def sai_nguoc_chieu(n, k):
    if (n & k) != n:                            # SAI: đòi n là tập con của k
        return -1
    return (n ^ k).bit_count()


for n, k in [(13, 4), (4, 13), (21, 21)]:
    print((n, k), "sai:", sai_nguoc_chieu(n, k), "| đúng:", Solution().minChanges(n, k))
assert sai_nguoc_chieu(13, 4) != Solution().minChanges(13, 4)

(13, 4) sai: -1 | đúng: 2
(4, 13) sai: 2 | đúng: -1
(21, 21) sai: 0 | đúng: 0


**Vì sao sai:** ta biến `n` về `k` bằng cách **xóa** bit, nên `k` phải là tập con của `n` (không phải ngược lại). Với `(13, 4)`, `13 & 4 = 4 != 13` nên code sai trả `-1`. **Cách sửa:** `n & k == k`.

### Lỗi 3: Dùng hiệu số bit `1` (`popcount(n) − popcount(k)`)

In [25]:
def sai_hieu_popcount2(n, k):
    return n.bit_count() - k.bit_count()        # SAI: chỉ so số lượng, bỏ qua vị trí


print("sai:", sai_hieu_popcount2(14, 13), "| đúng:", Solution().minChanges(14, 13))      # 0 vs -1
print("sai:", sai_hieu_popcount2(13, 4), "| đúng:", Solution().minChanges(13, 4))        # 1 vs 2

sai: 0 | đúng: -1
sai: 2 | đúng: 2


**Vì sao sai:** số bit `1` bằng nhau không có nghĩa là cùng vị trí. `14` và `13` đều có `3` bit `1` nhưng khác vị trí, nên công thức sai cho `0` thay vì `-1`. Với ví dụ 1 (`13` có `3` bit, `4` có `1` bit, hiệu `2`) công thức sai lại **trùng** đáp án đúng — đây là kiểu lỗi "đúng ví dụ mẫu nhưng sai bài". **Cách sửa:** so **vị trí** bằng `n ^ k` và `n & k`.

### Lỗi 4: Ưu tiên toán tử trong ngôn ngữ khác (C/Java) — luôn thêm ngoặc

In [26]:
n, k = 13, 4
print((n & k) != k)         # False: có ngoặc, rõ ràng
print(n & k != k)           # Python: & ưu tiên hơn !=, cho cùng kết quả
# Trong C/C++/Java, `n & k != k` được hiểu là n & (k != k) -> SAI. Thói quen tốt: LUÔN viết (n & k) != k.
assert ((n & k) != k) == (n & k != k)

False
False


**Lưu ý:** trong Python `&` ưu tiên hơn `!=` nên viết thiếu ngoặc vẫn đúng; nhưng khi chuyển sang C++/Java thì **sai**. Hãy tập thói quen thêm ngoặc.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Chỉ được đổi `1 → 0`**" ⇒ kết quả là **tập con**: `n & k == k`. Chỉ được đổi `0 → 1` ⇒ `n | k == k`.
- "**Số bit khác nhau**" ⇒ `popcount(a ^ b)` (khoảng cách Hamming).
- Khi `k` là tập con của `n`: `n ^ k == n − k == n & ~k`, nên có thể viết `(n - k).bit_count()` (ngắn, nhưng chỉ đúng khi đã kiểm khả thi).
- Đếm bit `1`: `bin(x).count("1")` hoặc `x.bit_count()`; thi cử không cho `bit_count` thì dùng `x &= x - 1`.
- Bài bit: luôn **viết nhị phân ra giấy và xếp thẳng cột** trước khi nghĩ công thức.

## 15. Biến thể của bài

1. **Cho phép đổi cả hai chiều** (`0 ↔ 1`): đáp án luôn tồn tại và bằng `popcount(n ^ k)` (**Hamming Distance, 461** / **Minimum Bit Flips to Convert Number, 2220**).
2. **Chỉ được đổi `0 → 1`**: điều kiện `n | k == k`, đáp án `popcount(n ^ k)`.
3. **Đếm số `k` đạt được từ `n`**: mỗi bit `1` của `n` có thể giữ hoặc xóa ⇒ `2^popcount(n)` giá trị (kể cả `0` và `n`).
4. **Liệt kê** mọi `k` đạt được: duyệt submask `s = (s - 1) & n`.
5. Bài liên quan: **Number of 1 Bits (191)**, **Counting Bits (338)**, **Power of Two (231)**, **Single Number (136)**.

In [27]:
def hamming_both_ways(n, k):
    return (n ^ k).bit_count()


def only_zero_to_one(n, k):
    return -1 if (n | k) != k else (n ^ k).bit_count()


def count_reachable(n):
    return 2 ** n.bit_count()


def list_reachable(n):
    result = []
    s = n
    while True:
        result.append(s)
        if s == 0:
            break
        s = (s - 1) & n               # submask kế tiếp theo thứ tự giảm dần
    return sorted(result)


print(hamming_both_ways(14, 13))                 # 2
print(only_zero_to_one(4, 13), only_zero_to_one(13, 4))   # 2 -1
print(count_reachable(13), list_reachable(13))   # 8 [0, 1, 4, 5, 8, 9, 12, 13]
assert hamming_both_ways(14, 13) == 2
assert only_zero_to_one(4, 13) == 2 and only_zero_to_one(13, 4) == -1
assert count_reachable(13) == 8
assert list_reachable(13) == [0, 1, 4, 5, 8, 9, 12, 13]
assert all(Solution().minChanges(13, s) != -1 for s in list_reachable(13) if s > 0)

2
2 -1
8 [0, 1, 4, 5, 8, 9, 12, 13]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Từng bit | xét 20 cột `(n_i, k_i)` | `O(log n)` | `O(1)` | học, bám sát đề |
| 2. `bin().count("1")` | `n & k == k` rồi đếm `'1'` của `n ^ k` | `O(log n)` | `O(log n)` | Python cũ, không có `bit_count` |
| 3. `bit_count()` | giống Cách 2, đếm bằng hàm dựng sẵn | `O(1)` thực tế | `O(1)` | **nên nộp** |
| 4. Kernighan | `x &= x - 1` đếm số vòng | `O(p)` | `O(1)` | phỏng vấn, không dùng hàm dựng sẵn |
| 5. BFS | mô phỏng đúng luật từng bước | `O(2^q · log n)` | `O(2^q)` | **chỉ để kiểm chứng** với `n` nhỏ |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `bin(10)`, `10 & 2`, `10 ^ 2`, `(10 ^ 2).bit_count()`, `Solution().minChanges(10, 2)`, `Solution().minChanges(6, 1)`.

**Bài 2 (tính tay).** Với `n = 29` và `k = 9`: viết nhị phân thẳng cột (5 bit), kiểm tra `n & k == k`, tính `n ^ k` rồi đếm bit `1`; cho biết đáp án.

**Bài 3 (điền TODO).** Hoàn thành `min_changes_todo(n, k)` theo Cách 4 (Kernighan).

In [28]:
def min_changes_todo(n, k):
    # TODO 1: nếu (n & k) != k thì trả về -1
    # TODO 2: x = n ^ k; count = 0
    # TODO 3: while x: x &= x - 1; count += 1
    # TODO 4: trả về count
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `min_changes_shift(n, k)` **không dùng** `bin`, `bit_count`, `&= x - 1`: chỉ dùng dịch bit — lặp `while n or k:` xét bit thấp nhất `n & 1` và `k & 1`, rồi `n >>= 1; k >>= 1`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `count_reachable(n)` đếm số giá trị `k` (kể cả `0` và `n`) có thể đạt được từ `n`. (b) Viết `farthest(n)` trả về số lần đổi **lớn nhất** có thể phải dùng từ `n` (tới `k = 1..n` đạt được). (c) Viết `min_changes_to_any(n, targets)` trả về số lần đổi nhỏ nhất trong các `k` thuộc danh sách `targets` mà `n` đạt được, hoặc `-1` nếu không `k` nào đạt được.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `10 = 1010`, `2 = 0010`; AND giữ bit chung, XOR giữ bit khác. `6 = 110`, `1 = 001`: bit thấp nhất của `k` là `1` mà của `n` là `0`. Bài 3: sao chép đúng thứ tự `if`, `x`, `while`.
- **Mức 2 (vừa):** Bài 2: `29 = 11101`, `9 = 01001` ⇒ `29 & 9 = 01001 = 9` (khả thi), `29 ^ 9 = 10100`. Bài 4: tại mỗi bước nếu `(n & 1, k & 1) == (0, 1)` thì trả `-1`; nếu `(1, 0)` thì đếm.
- **Mức 3 (sâu):** Bài 5(a): mỗi bit `1` của `n` giữ hoặc xóa ⇒ `2 ** n.bit_count()`. 5(b): với `k = 1` mà `1` là tập con của `n` (n lẻ) cần `popcount(n) − 1` lần; tổng quát duyệt mọi `k` trong `1..n` và lấy `max` của kết quả khác `-1`. 5(c): lọc `k` có `Solution().minChanges(n, k) != -1` rồi lấy `min`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [29]:
# Bài 1
assert bin(10) == "0b1010"
assert 10 & 2 == 2
assert 10 ^ 2 == 8
assert (10 ^ 2).bit_count() == 1
assert Solution().minChanges(10, 2) == 1
assert Solution().minChanges(6, 1) == -1

# Bài 2
n, k = 29, 9
assert format(n, "05b") == "11101" and format(k, "05b") == "01001"
assert n & k == k
assert format(n ^ k, "05b") == "10100"
assert (n ^ k).bit_count() == 2
assert Solution().minChanges(29, 9) == 2


# Bài 3
def min_changes_todo_ans(n, k):
    if (n & k) != k:
        return -1
    x = n ^ k
    count = 0
    while x:
        x &= x - 1
        count += 1
    return count


for n, k, expected in examples:
    assert min_changes_todo_ans(n, k) == expected


# Bài 4
def min_changes_shift(n, k):
    count = 0
    while n or k:
        a, b = n & 1, k & 1
        if a == 0 and b == 1:
            return -1
        if a == 1 and b == 0:
            count += 1
        n >>= 1
        k >>= 1
    return count


for n, k, expected in examples:
    assert min_changes_shift(n, k) == expected
random.seed(1)
for _ in range(500):
    n, k = random.randint(1, 10**6), random.randint(1, 10**6)
    assert min_changes_shift(n, k) == Solution().minChanges(n, k)


# Bài 5
def count_reachable_ans(n):
    return 2 ** n.bit_count()


def farthest(n):
    best = 0
    for k in range(1, n + 1):
        r = Solution().minChanges(n, k)
        if r != -1:
            best = max(best, r)
    return best


def min_changes_to_any(n, targets):
    results = [Solution().minChanges(n, t) for t in targets]
    results = [r for r in results if r != -1]
    return min(results) if results else -1


assert count_reachable_ans(13) == 8 and count_reachable_ans(16) == 2
assert farthest(13) == 2 and farthest(15) == 3 and farthest(1) == 0
assert min_changes_to_any(13, [2, 6, 12]) == 1       # 13 -> 12 chỉ đổi 1 bit
assert min_changes_to_any(13, [2, 6]) == -1
assert min_changes_to_any(13, [4, 5, 9]) == 1
print("PASS toàn bộ đáp án")

PASS toàn bộ đáp án


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (chỉ được xóa bit `1` ⇒ `k` phải là tập con của `n`; đáp án là số bit khác nhau)

```python
class Solution:
    def minChanges(self, n: int, k: int) -> int:
        if (n & k) != k:
            return -1
        return (n ^ k).bit_count()
```

**Checklist**

- [ ] Đã kiểm tra `(n & k) != k` **trước** khi đếm (tập con theo đúng chiều: `k` ⊆ `n`)?
- [ ] Đã dùng `n ^ k` (bit khác nhau) rồi đếm bit `1`?
- [ ] Có ngoặc quanh `n & k` để tránh nhầm ưu tiên toán tử?
- [ ] Có dự phòng nếu `bit_count` không có: `bin(x).count("1")` hoặc `x &= x - 1`?
- [ ] Đã thử `(13, 4)` ⇒ `2`, `(21, 21)` ⇒ `0`, `(14, 13)` ⇒ `-1`, và `(8, 7)` ⇒ `-1`?
- [ ] Skeleton đúng `class Solution` / `def minChanges(self, n: int, k: int) -> int`?